# Enrichment analysis (R), discovery tier (BH<0.05 hits)

ORA + GSEA against multiple gene-set collections for the discovery tier: all genes with `padj_treatment < 0.05` from the primary main-effect screen (211 inhibitory, 35 excitatory).

The same analyses on the bootstrap-robust subset of hits give the robust tier (`enrichment_robust`). The figures use the discovery tier.

## Design choices

- Foreground: `padj_treatment < 0.05` per lineage (211 inh, 35 exc).
- Background universe: post-DRIMSeq-filter gene set (5,441 inh / 7,544 exc). Using the full transcriptome instead would inflate enrichment for "any expressed brain gene" terms.
- GSEA ranking: `-log10(pvalue_treatment)`, descending. Unsigned by Δψ: Δψ is a compositional centroid shift, not a quantity that maps to protein-level activity (longer/shorter UTR can either stabilise or destabilise the transcript). A signed GSEA would therefore be misleading.
- Term-size filter: min 10, max 500 genes per term.
- FDR: BH within each collection.

## Visualisation: ridgeplots, not dotplots

For each enriched term (BH<0.05) we show a density ridge of `-log10(pvalue_treatment)` across all the term's members that appear in the gene universe, not just the foreground hits. This shows whether the enrichment is systematic (most term members shifted toward strong p-values) or driven by a few genes (a few low-p tails on a flat background). A vertical dashed line marks the nominal `p_value_treatment = 0.05` threshold for reference.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
  REPO_ROOT <- normalizePath(getwd())
  while (!file.exists(file.path(REPO_ROOT, "environment", "apa-python.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
    REPO_ROOT <- dirname(REPO_ROOT)
  }
  Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR <- file.path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR_treatment")
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE)
setwd(WORK_DIR)

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(stringr)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(DOSE)
  library(enrichplot)
  library(ReactomePA)
  library(msigdbr)
  library(ggplot2)
  library(ggridges)
  library(viridis)
})
options(repr.plot.width = 9, repr.plot.height = 6, repr.plot.res = 110)
cat("Setup complete.\n")

In [ ]:
LINEAGES <- c("inhibitory", "excitatory")
PADJ_THRESHOLD <- 0.05
MIN_GS_SIZE <- 10L
MAX_GS_SIZE <- 500L

RESULTS_ROOT <- paste0(DATA_ROOT, "/3utr/organoids/knoblich/utrcalling/dirmult_treatment_results")
SFARI_CSV <- paste0(DATA_ROOT, "/common_files/annotations/SFARI-Gene_genes_03-22-2026release_03-27-2026export.csv")

# Tier-specific suffix ("robust" for the bootstrap-robust subset)
TIER <- "discovery"
metrics_csv <- function(ln) file.path(RESULTS_ROOT, ln, "calculate_metrics", "metrics_per_gene.csv")
out_dir <- function(ln) {
  d <- file.path(RESULTS_ROOT, ln, paste0("enrichment_", TIER))
  dir.create(d, showWarnings = FALSE, recursive = TRUE)
  fig_d <- file.path(d, "figures")
  dir.create(fig_d, showWarnings = FALSE, recursive = TRUE)
  list(root = d, figures = fig_d)
}
for (ln in LINEAGES) {
  od <- out_dir(ln)
  cat(sprintf("  %-12s csv: %s\n", ln, metrics_csv(ln)))
  cat(sprintf("  %-12s out: %s\n", "", od$root))
}

## Step 1: load + define discovery foreground per lineage

In [ ]:
load_and_prep <- function(lineage) {
  cat(sprintf("\n[%s] loading ...\n", lineage))
  m <- readr::read_csv(metrics_csv(lineage), show_col_types = FALSE)
  m$gene_id <- str_trim(sub("^\\|+", "", m$gene_id))
  m <- m %>% filter(!grepl("\\|", gene_id), !grepl(",", gene_id))
  m$entrezid <- mapIds(org.Hs.eg.db,
    keys = m$gene_id,
    column = "ENTREZID", keytype = "SYMBOL", multiVals = "first"
  )
  n_mapped <- sum(!is.na(m$entrezid))
  cat(sprintf(
    "  rows: %d ; ENTREZ-mapped: %d (%.1f%%)\n",
    nrow(m), n_mapped, 100 * n_mapped / nrow(m)
  ))
  p <- m$pvalue_treatment
  p[!is.finite(p)] <- NA
  if (any(p == 0, na.rm = TRUE)) p[p == 0] <- min(p[p > 0], na.rm = TRUE)
  m$rank_score <- -log10(p)
  # Discovery foreground = padj_treatment < threshold
  m$is_hit <- !is.na(m$padj_treatment) & m$padj_treatment < PADJ_THRESHOLD
  cat(sprintf("  DISCOVERY foreground (padj<%s): %d\n", PADJ_THRESHOLD, sum(m$is_hit)))
  m
}
dm <- list()
for (ln in LINEAGES) dm[[ln]] <- load_and_prep(ln)

## Step 2: pre-load gene set collections

In [ ]:
cat("Loading MSigDB collections ...\n")
hallmark_t2g <- msigdbr(species = "Homo sapiens", collection = "H") |>
  dplyr::select(gs_name, gene_symbol) |>
  distinct()
hpo_t2g <- msigdbr(species = "Homo sapiens", collection = "C5", subcollection = "HPO") |>
  dplyr::select(gs_name, gene_symbol) |>
  distinct()
cat(sprintf("  Hallmark: %d sets\n", n_distinct(hallmark_t2g$gs_name)))
cat(sprintf("  HPO:      %d sets\n", n_distinct(hpo_t2g$gs_name)))

In [ ]:
sfari <- readr::read_csv(SFARI_CSV, show_col_types = FALSE)
sfari_score1 <- sfari %>%
  filter(`gene-score` == 1) %>%
  pull(`gene-symbol`) %>%
  unique() %>%
  na.omit()
sfari_score12 <- sfari %>%
  filter(`gene-score` %in% c(1, 2)) %>%
  pull(`gene-symbol`) %>%
  unique() %>%
  na.omit()
sfari_syndromic <- sfari %>%
  filter(syndromic == 1) %>%
  pull(`gene-symbol`) %>%
  unique() %>%
  na.omit()
sfari_t2g <- bind_rows(
  tibble(gs_name = "SFARI_score1", gene_symbol = sfari_score1),
  tibble(gs_name = "SFARI_score1_or_2", gene_symbol = sfari_score12),
  tibble(gs_name = "SFARI_syndromic", gene_symbol = sfari_syndromic)
)
cat(sprintf(
  "  SFARI: score1=%d, score1_or_2=%d, syndromic=%d\n",
  length(sfari_score1), length(sfari_score12), length(sfari_syndromic)
))

In [ ]:
APA_MACHINERY_GENES <- unique(c(
  "CPSF1", "CPSF2", "CPSF3", "CPSF3L", "CPSF4", "CPSF6", "CPSF7",
  "NUDT21", "CSTF1", "CSTF2", "CSTF2T", "CSTF3", "CLP1", "PCF11",
  "PAPOLA", "PAPOLB", "PAPOLG", "PABPN1", "PABPC1", "PABPC4", "SYMPK",
  "WDR33", "FIP1L1", "RBBP6", "XRN2", "SETX", "ELAVL1", "HNRNPK"
))
apa_t2g <- tibble(gs_name = "APA_MACHINERY", gene_symbol = APA_MACHINERY_GENES)
cat(sprintf("  APA-machinery: %d genes\n", length(APA_MACHINERY_GENES)))

rbp_entrez <- AnnotationDbi::select(org.Hs.eg.db,
  keys = "GO:0003723",
  columns = c("SYMBOL"), keytype = "GOALL"
) %>%
  pull(SYMBOL) %>%
  unique() %>%
  na.omit()
rbp_t2g <- tibble(gs_name = "RBP_GO0003723", gene_symbol = rbp_entrez)
cat(sprintf("  RBP set: %d gene symbols\n", length(rbp_entrez)))

## Step 3: per-lineage analysis (stores enrichResult objects for ridgeplots)

In [ ]:
# ORA / GSEA helpers: return both the data.frame and the enrichResult object
# so we can access @geneSets for the ridgeplot of all term members.
ora_t2g <- function(fg_sym, universe_sym, t2g, name, min_gs = MIN_GS_SIZE, max_gs = MAX_GS_SIZE) {
  res <- tryCatch(
    clusterProfiler::enricher(
      gene = fg_sym, universe = universe_sym, TERM2GENE = t2g,
      pAdjustMethod = "BH", pvalueCutoff = 1.0, qvalueCutoff = 1.0,
      minGSSize = min_gs, maxGSSize = max_gs
    ),
    error = function(e) {
      message("  ORA[", name, "] error: ", conditionMessage(e))
      NULL
    }
  )
  if (is.null(res)) {
    return(list(df = data.frame(), obj = NULL))
  }
  df <- as.data.frame(res)
  if (nrow(df) == 0) {
    return(list(df = df, obj = res))
  }
  df$collection <- name
  list(df = df, obj = res)
}

gsea_t2g <- function(v_sym, t2g, name, min_gs = MIN_GS_SIZE, max_gs = MAX_GS_SIZE) {
  res <- tryCatch(
    clusterProfiler::GSEA(
      geneList = sort(v_sym, decreasing = TRUE), TERM2GENE = t2g,
      pAdjustMethod = "BH", pvalueCutoff = 1.0,
      minGSSize = min_gs, maxGSSize = max_gs, scoreType = "pos"
    ),
    error = function(e) {
      message("  GSEA[", name, "] error: ", conditionMessage(e))
      NULL
    }
  )
  if (is.null(res)) {
    return(list(df = data.frame(), obj = NULL))
  }
  df <- as.data.frame(res)
  if (nrow(df) == 0) {
    return(list(df = df, obj = res))
  }
  df$collection <- name
  list(df = df, obj = res)
}

In [ ]:
# Helper to get gene-set members as symbols (handles ENTREZ-keyed sets too)
get_gene_sets_symbol <- function(enrich_obj) {
  if (is.null(enrich_obj)) {
    return(NULL)
  }
  gs <- enrich_obj@geneSets
  if (length(gs) == 0) {
    return(NULL)
  }
  first_members <- head(unlist(gs[1]), 5)
  is_entrez <- length(first_members) > 0 && all(grepl("^[0-9]+$", first_members))
  if (is_entrez) {
    all_ent <- unique(unlist(gs))
    sym <- mapIds(org.Hs.eg.db,
      keys = all_ent, column = "SYMBOL",
      keytype = "ENTREZID", multiVals = "first"
    )
    sym_map <- setNames(as.character(sym), all_ent)
    gs <- lapply(gs, function(x) {
      mapped <- unname(sym_map[as.character(x)])
      mapped[!is.na(mapped) & mapped != ""]
    })
  }
  gs
}

In [ ]:
run_lineage <- function(lineage) {
  m <- dm[[lineage]]
  od <- out_dir(lineage)
  cat(sprintf("\n========= %s ==========\n", lineage))
  fg_sym <- m %>%
    filter(is_hit) %>%
    pull(gene_id) %>%
    unique()
  universe_sym <- unique(m$gene_id)
  fg_entrez <- m %>%
    filter(is_hit) %>%
    pull(entrezid) %>%
    na.omit() %>%
    unique() %>%
    as.character()
  universe_entrez <- m %>%
    pull(entrezid) %>%
    na.omit() %>%
    unique() %>%
    as.character()
  cat(sprintf(
    "  foreground: %d sym / %d entrez | universe: %d sym / %d entrez\n",
    length(fg_sym), length(fg_entrez), length(universe_sym), length(universe_entrez)
  ))
  if (length(fg_sym) < 5) {
    cat("  WARN: fewer than 5 foreground genes -- skipping.\n")
    return(NULL)
  }
  v_sym <- setNames(m$rank_score, m$gene_id)
  v_sym <- v_sym[!is.na(v_sym) & !is.na(names(v_sym)) & names(v_sym) != ""]
  v_sym <- v_sym[!duplicated(names(v_sym))]
  v_ent <- setNames(m$rank_score, m$entrezid)
  v_ent <- v_ent[!is.na(v_ent) & !is.na(names(v_ent))]
  v_ent <- v_ent[!duplicated(names(v_ent))]

  ora <- list()
  gsea <- list()
  ora_obj <- list()
  gsea_obj <- list()

  cat("  GO BP ...\n")
  go_o <- tryCatch(
    clusterProfiler::enrichGO(
      gene = fg_sym, universe = universe_sym, OrgDb = org.Hs.eg.db,
      keyType = "SYMBOL", ont = "BP", pAdjustMethod = "BH",
      pvalueCutoff = 1.0, qvalueCutoff = 1.0,
      minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE
    ),
    error = function(e) NULL
  )
  if (!is.null(go_o) && nrow(as.data.frame(go_o)) > 0) {
    df <- as.data.frame(go_o)
    df$collection <- "GO_BP"
    ora[["GO_BP"]] <- df
    ora_obj[["GO_BP"]] <- go_o
    cat(sprintf(
      "    GO BP ORA: %d sig (padj<0.05) of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }
  go_g <- tryCatch(clusterProfiler::gseGO(
    geneList = sort(v_sym, decreasing = TRUE), OrgDb = org.Hs.eg.db,
    keyType = "SYMBOL", ont = "BP", pAdjustMethod = "BH",
    pvalueCutoff = 1.0, minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE,
    scoreType = "pos"
  ), error = function(e) NULL)
  if (!is.null(go_g) && nrow(as.data.frame(go_g)) > 0) {
    df <- as.data.frame(go_g)
    df$collection <- "GO_BP"
    gsea[["GO_BP"]] <- df
    gsea_obj[["GO_BP"]] <- go_g
    cat(sprintf(
      "    GO BP GSEA: %d sig of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }

  for (def in list(
    list(name = "Hallmark", t2g = hallmark_t2g),
    list(name = "HPO", t2g = hpo_t2g),
    list(name = "SFARI", t2g = sfari_t2g, min_gs = 1, max_gs = 5000),
    list(name = "RBP", t2g = rbp_t2g, min_gs = 1, max_gs = 5000),
    list(name = "APA_machinery", t2g = apa_t2g, min_gs = 1, max_gs = 5000)
  )) {
    cat(sprintf("  %s ...\n", def$name))
    min_gs <- if (!is.null(def$min_gs)) def$min_gs else MIN_GS_SIZE
    max_gs <- if (!is.null(def$max_gs)) def$max_gs else MAX_GS_SIZE
    o <- ora_t2g(fg_sym, universe_sym, def$t2g, def$name, min_gs, max_gs)
    g <- gsea_t2g(v_sym, def$t2g, def$name, min_gs, max_gs)
    if (nrow(o$df) > 0) {
      ora[[def$name]] <- o$df
      ora_obj[[def$name]] <- o$obj
    }
    if (nrow(g$df) > 0) {
      gsea[[def$name]] <- g$df
      gsea_obj[[def$name]] <- g$obj
    }
  }

  cat("  KEGG ...\n")
  k_o <- tryCatch(
    clusterProfiler::enrichKEGG(
      gene = fg_entrez, universe = universe_entrez, organism = "hsa",
      pAdjustMethod = "BH", pvalueCutoff = 1.0, qvalueCutoff = 1.0,
      minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE
    ),
    error = function(e) NULL
  )
  if (!is.null(k_o) && nrow(as.data.frame(k_o)) > 0) {
    df <- as.data.frame(k_o)
    df$collection <- "KEGG"
    ora[["KEGG"]] <- df
    ora_obj[["KEGG"]] <- k_o
    cat(sprintf(
      "    KEGG ORA: %d sig of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }
  k_g <- tryCatch(
    clusterProfiler::gseKEGG(
      geneList = sort(v_ent, decreasing = TRUE), organism = "hsa",
      pAdjustMethod = "BH", pvalueCutoff = 1.0,
      minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE, scoreType = "pos"
    ),
    error = function(e) NULL
  )
  if (!is.null(k_g) && nrow(as.data.frame(k_g)) > 0) {
    df <- as.data.frame(k_g)
    df$collection <- "KEGG"
    gsea[["KEGG"]] <- df
    gsea_obj[["KEGG"]] <- k_g
    cat(sprintf(
      "    KEGG GSEA: %d sig of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }

  cat("  Reactome ...\n")
  r_o <- tryCatch(
    ReactomePA::enrichPathway(
      gene = fg_entrez, universe = universe_entrez, organism = "human",
      pAdjustMethod = "BH", pvalueCutoff = 1.0, qvalueCutoff = 1.0,
      minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE, readable = TRUE
    ),
    error = function(e) NULL
  )
  if (!is.null(r_o) && nrow(as.data.frame(r_o)) > 0) {
    df <- as.data.frame(r_o)
    df$collection <- "Reactome"
    ora[["Reactome"]] <- df
    ora_obj[["Reactome"]] <- r_o
    cat(sprintf(
      "    Reactome ORA: %d sig of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }
  r_g <- tryCatch(
    ReactomePA::gsePathway(
      geneList = sort(v_ent, decreasing = TRUE), organism = "human",
      pAdjustMethod = "BH", pvalueCutoff = 1.0,
      minGSSize = MIN_GS_SIZE, maxGSSize = MAX_GS_SIZE, scoreType = "pos"
    ),
    error = function(e) NULL
  )
  if (!is.null(r_g) && nrow(as.data.frame(r_g)) > 0) {
    df <- as.data.frame(r_g)
    df$collection <- "Reactome"
    gsea[["Reactome"]] <- df
    gsea_obj[["Reactome"]] <- r_g
    cat(sprintf(
      "    Reactome GSEA: %d sig of %d tested\n",
      sum(df$p.adjust < 0.05), nrow(df)
    ))
  }

  # Persist CSVs and RDS (the RDS files are reloaded in Step 3b)
  for (nm in names(ora)) readr::write_csv(ora[[nm]], file.path(od$root, sprintf("ora_%s.csv", nm)))
  for (nm in names(gsea)) readr::write_csv(gsea[[nm]], file.path(od$root, sprintf("gsea_%s.csv", nm)))
  saveRDS(
    setNames(m$rank_score, m$gene_id),
    file.path(od$root, sprintf("score_map_%s.rds", lineage))
  )
  saveRDS(
    list(ora = ora, gsea = gsea, ora_obj = ora_obj, gsea_obj = gsea_obj),
    file.path(od$root, sprintf("enrichment_objects_%s.rds", lineage))
  )

  list(
    ora = ora, gsea = gsea, ora_obj = ora_obj, gsea_obj = gsea_obj,
    lineage = lineage, od = od, score_map = setNames(m$rank_score, m$gene_id)
  )
}

results <- list()
for (ln in LINEAGES) results[[ln]] <- run_lineage(ln)

## Step 3b: load-back from RDS (skip heavy run_lineage)

After a kernel restart, run the setup cells (libraries + `LINEAGES`/`out_dir`/`metrics_csv` definitions) and then this section to reload everything that Step 3 computed, without re-running ORA + GSEA across all collections (which takes ~5 min). Step 4 (ridgeplots) and beyond work the same way afterwards.

The cell below reads the saved bundle `enrichment_objects_<lineage>.rds` and the per-gene `score_map_<lineage>.rds` and reconstructs the same `results` list that Step 3 produces. If `score_map_<lineage>.rds` does not yet exist (older runs), it falls back to rebuilding `score_map` from `metrics_per_gene.csv` using the same recipe as `load_and_prep`.

In [ ]:
# Load-back: reload Step 3 output from disk so we don't need to re-run analyses.
# Requires: LINEAGES, out_dir(), metrics_csv() from setup cells.

results <- list()
for (ln in LINEAGES) {
  od <- out_dir(ln)
  bundle_fn <- file.path(od$root, sprintf("enrichment_objects_%s.rds", ln))
  score_fn <- file.path(od$root, sprintf("score_map_%s.rds", ln))
  if (!file.exists(bundle_fn)) {
    cat(sprintf(
      "  [%s] no cached enrichment RDS at %s -- run Step 3 first\n",
      ln, bundle_fn
    ))
    next
  }
  bundle <- readRDS(bundle_fn)
  if (file.exists(score_fn)) {
    score_map <- readRDS(score_fn)
  } else {
    # Older runs did not save score_map; rebuild from metrics CSV.
    m <- readr::read_csv(metrics_csv(ln), show_col_types = FALSE)
    m$gene_id <- stringr::str_trim(sub("^\\|+", "", m$gene_id))
    m <- m %>% filter(!grepl("\\|", gene_id), !grepl(",", gene_id))
    p <- m$pvalue_treatment
    p[!is.finite(p)] <- NA
    if (any(p == 0, na.rm = TRUE)) p[p == 0] <- min(p[p > 0], na.rm = TRUE)
    m$rank_score <- -log10(p)
    score_map <- setNames(m$rank_score, m$gene_id)
  }
  results[[ln]] <- list(
    ora = bundle$ora, gsea = bundle$gsea,
    ora_obj = bundle$ora_obj, gsea_obj = bundle$gsea_obj,
    lineage = ln, od = od, score_map = score_map
  )
  cat(sprintf(
    "  [%s] loaded: %d ORA + %d GSEA collections; score_map: %d genes\n",
    ln, length(results[[ln]]$ora), length(results[[ln]]$gsea),
    length(score_map)
  ))
}

## Step 4: ridgeplots (per term, per collection)

For each enriched term (padj < 0.05), plot a density ridge of `-log10(pvalue_treatment)` across all gene-set members that are in our universe. Vertical dashed line = nominal p=0.05 threshold for orientation. The wider the high-p tail, the more systematic the enrichment (i.e. many term members affected, not just a handful).

In [ ]:
save_ridgeplot <- function(res_df, gene_sets_sym, score_map, name, kind, lineage, od,
                           n_top = 20, padj_cut = 0.05, min_members = 3) {
  if (is.null(res_df) || nrow(res_df) == 0 || is.null(gene_sets_sym)) {
    return(invisible(NULL))
  }
  sig <- res_df %>%
    filter(p.adjust < padj_cut) %>%
    arrange(p.adjust) %>%
    head(n_top)
  if (nrow(sig) == 0) {
    return(invisible(NULL))
  }
  rows <- list()
  for (i in seq_len(nrow(sig))) {
    term_id <- sig$ID[i]
    desc <- sig$Description[i]
    padj <- sig$p.adjust[i]
    members <- gene_sets_sym[[term_id]]
    if (is.null(members) || length(members) == 0) next
    members_in_uni <- intersect(members, names(score_map))
    if (length(members_in_uni) < min_members) next
    rows[[term_id]] <- data.frame(
      term_id = term_id,
      term_label = stringr::str_wrap(desc, 45),
      padj = padj,
      score = unname(score_map[members_in_uni]),
      stringsAsFactors = FALSE
    )
  }
  if (length(rows) == 0) {
    return(invisible(NULL))
  }
  df <- bind_rows(rows)
  df$term_label <- factor(df$term_label, levels = rev(unique(df$term_label)))
  df$neg_log_padj <- -log10(df$padj)
  vline_x <- -log10(0.05)
  fill_low <- -log10(padj_cut)
  fill_high <- max(df$neg_log_padj)
  if (fill_high - fill_low < 1) fill_high <- fill_low + 1
  p <- ggplot(df, aes(x = score, y = term_label, fill = neg_log_padj)) +
    geom_density_ridges(
      scale = 2.5, alpha = 0.7,
      quantile_lines = TRUE, quantiles = 2,
      color = "white", linewidth = 0.25
    ) +
    geom_vline(
      xintercept = vline_x, linetype = "dashed",
      color = "black", linewidth = 0.4
    ) +
    scale_fill_viridis(
      name = sprintf("%s\n-log10(p.adjust)", toupper(kind)),
      option = "plasma", direction = 1,
      limits = c(fill_low, fill_high), oob = scales::squish
    ) +
    xlab("-log10(pvalue_treatment) per term-member gene (universe-restricted)") +
    ylab(NULL) +
    ggtitle(sprintf(
      "%s [%s] %s -- top %d enriched terms",
      name, lineage, toupper(kind), length(rows)
    )) +
    theme_minimal(base_size = 9) +
    theme(
      plot.title = element_text(size = 9),
      axis.text.y = element_text(size = 7)
    )
  height <- max(2.6, 0.28 * length(rows) + 1.4)
  ggsave(file.path(od$figures, sprintf("ridge_%s_%s.pdf", kind, name)),
    p,
    width = 9, height = height, dpi = 300
  )
  ggsave(file.path(od$figures, sprintf("ridge_%s_%s.png", kind, name)),
    p,
    width = 9, height = height, dpi = 200
  )
  p
}

# Render ridgeplots for both ORA and GSEA across all collections for each lineage
for (ln in LINEAGES) {
  if (is.null(results[[ln]])) next
  od <- results[[ln]]$od
  score_map <- results[[ln]]$score_map
  cat(sprintf("\n--- %s ridgeplots ---\n", ln))
  # ORA
  for (nm in names(results[[ln]]$ora_obj)) {
    gs <- get_gene_sets_symbol(results[[ln]]$ora_obj[[nm]])
    p <- save_ridgeplot(results[[ln]]$ora[[nm]], gs, score_map, nm, "ora", ln, od)
    if (!is.null(p)) {
      cat(sprintf("  ORA %s: ridge saved\n", nm))
      print(p)
    } else {
      cat(sprintf("  ORA %s: no sig terms or insufficient members\n", nm))
    }
  }
  # GSEA
  for (nm in names(results[[ln]]$gsea_obj)) {
    gs <- get_gene_sets_symbol(results[[ln]]$gsea_obj[[nm]])
    p <- save_ridgeplot(results[[ln]]$gsea[[nm]], gs, score_map, nm, "gsea", ln, od)
    if (!is.null(p)) {
      cat(sprintf("  GSEA %s: ridge saved\n", nm))
      print(p)
    } else {
      cat(sprintf("  GSEA %s: no sig terms or insufficient members\n", nm))
    }
  }
}

## Step 5: top-terms summary CSVs + console table

In [ ]:
summarise_top <- function(res, kind = "ora", n_top = 10) {
  pieces <- list()
  for (col_nm in names(res[[kind]])) {
    df <- res[[kind]][[col_nm]]
    if (nrow(df) == 0) next
    df <- df %>%
      arrange(p.adjust) %>%
      head(n_top) %>%
      mutate(lineage = res$lineage, kind = toupper(kind))
    keep <- intersect(c(
      "lineage", "kind", "collection", "ID", "Description", "p.adjust",
      "GeneRatio", "BgRatio", "NES", "setSize"
    ), names(df))
    pieces[[col_nm]] <- df[, keep, drop = FALSE]
  }
  if (length(pieces) == 0) {
    return(data.frame())
  }
  bind_rows(pieces)
}

top_ora <- bind_rows(lapply(results, summarise_top, kind = "ora"))
top_gsea <- bind_rows(lapply(results, summarise_top, kind = "gsea"))
write_csv(top_ora, file.path(RESULTS_ROOT, sprintf("enrichment_%s_top_ora.csv", TIER)))
write_csv(top_gsea, file.path(RESULTS_ROOT, sprintf("enrichment_%s_top_gsea.csv", TIER)))

cat("\n=== TOP-10 ORA terms (per lineage, per collection) ===\n")
print(as.data.frame(top_ora))
cat("\n=== TOP-10 GSEA terms (per lineage, per collection) ===\n")
print(as.data.frame(top_gsea))

## Step 6: SFARI, HPO-autism, APA-machinery focus

In [ ]:
print_df <- function(df, cols_keep = NULL, max_rows = 50) {
  df <- as.data.frame(df)
  if (!is.null(cols_keep)) {
    keep <- intersect(cols_keep, names(df))
    df <- df[, keep, drop = FALSE]
  }
  if (nrow(df) > max_rows) df <- head(df, max_rows)
  rownames(df) <- NULL
  print(df)
}

for (ln in LINEAGES) {
  cat(sprintf("\n--- %s focus ---\n", ln))
  if (is.null(results[[ln]])) next
  for (kind in c("ora", "gsea")) {
    df <- results[[ln]][[kind]][["SFARI"]]
    if (!is.null(df) && nrow(df) > 0) {
      cat(sprintf("\n  SFARI %s:\n", toupper(kind)))
      df <- df[order(df$p.adjust), ]
      print_df(df, c("ID", "p.adjust", "Count", "GeneRatio", "BgRatio", "NES", "setSize"))
    }
  }
  df_h <- results[[ln]][["ora"]][["HPO"]]
  if (!is.null(df_h) && nrow(df_h) > 0) {
    autism_hits <- df_h[grepl("AUTISM|AUTISTIC|INTELLECT", df_h$ID, ignore.case = TRUE), ]
    if (nrow(autism_hits) > 0) {
      cat(sprintf("\n  HPO ORA autism-related (n=%d):\n", nrow(autism_hits)))
      autism_hits <- autism_hits[order(autism_hits$p.adjust), ]
      print_df(autism_hits, c("ID", "p.adjust", "Count", "GeneRatio"), max_rows = 15)
    }
  }
  for (kind in c("ora", "gsea")) {
    df <- results[[ln]][[kind]][["APA_machinery"]]
    if (!is.null(df) && nrow(df) > 0) {
      cat(sprintf("\n  APA_machinery %s:\n", toupper(kind)))
      print_df(df)
    }
  }
  for (kind in c("ora", "gsea")) {
    df <- results[[ln]][[kind]][["RBP"]]
    if (!is.null(df) && nrow(df) > 0) {
      cat(sprintf("\n  RBP %s:\n", toupper(kind)))
      print_df(df)
    }
  }
}

## Step 7: SFARI score-stratified cross-lineage panel CSV

In [ ]:
sfari_panel_rows <- list()
for (ln in LINEAGES) {
  if (is.null(results[[ln]])) next
  for (kind in c("ora", "gsea")) {
    df <- results[[ln]][[kind]][["SFARI"]]
    if (is.null(df) || nrow(df) == 0) next
    df$lineage <- ln
    df$kind <- toupper(kind)
    sfari_panel_rows[[paste(ln, kind, sep = "_")]] <- df
  }
}
sfari_panel <- bind_rows(sfari_panel_rows)
if (nrow(sfari_panel) > 0) {
  write_csv(sfari_panel, file.path(RESULTS_ROOT, sprintf("enrichment_%s_sfari_panel.csv", TIER)))
  cat("\nSFARI panel rows:\n")
  keep <- intersect(
    c("lineage", "kind", "ID", "p.adjust", "Count", "GeneRatio", "NES", "setSize"),
    names(sfari_panel)
  )
  sfari_df <- as.data.frame(sfari_panel[, keep, drop = FALSE])
  rownames(sfari_df) <- NULL
  print(sfari_df)
}